In [34]:
import numpy as np
import pandas as pd

train = pd.read_csv('phone-addiction-train.csv')
test = pd.read_csv('phone-addiction-test.csv')

print("train shape", train.shape)
print("test shape", test.shape)
test.head()

train shape (691369, 14)
test shape (296302, 13)


,id,age,daily_screen_time_hours,social_media_hours,gaming_hours,work_study_hours,sleep_hours,notifications_per_day,app_opens_per_day,weekend_screen_time,gender,stress_level,academic_work_impact
0,691369,30.0,9.34,NaN,0.77,4.09,7.15,153.0,16.0,NaN,Other,Medium,Yes
1,691370,NaN,NaN,1.91,NaN,1.20,8.67,239.0,148.0,10.68,Female,High,No
2,691371,26.0,8.48,3.64,NaN,3.39,7.47,106.0,123.0,NaN,Male,NaN,No
3,691372,20.0,8.37,2.99,1.69,2.53,5.45,178.0,55.0,9.88,Male,High,Yes
4,691373,25.0,8.25,4.02,1.21,2.32,8.46,63.0,86.0,10.72,Male,Low,No


In [2]:
#Find NullValues
def MissingSummary(x):
    return pd.DataFrame({"MissingCount" : x.isna().sum(),
                         "MissingMean" : x.isna().mean()})

TrainMissing = MissingSummary(train.drop(columns=['addicted_label', 'id']))
TestMissing = MissingSummary(test.drop(columns=['id']))

missing = pd.concat([TrainMissing.add_prefix('Train'), TestMissing.add_prefix('Test')], axis=1)
missing = missing.sort_values(by="TrainMissingMean", ascending=False)
missing.round(2)

,TrainMissingCount,TrainMissingMean,TestMissingCount,TestMissingMean
social_media_hours,133995,0.19,47397,0.16
gaming_hours,126821,0.18,59420,0.20
weekend_screen_time,112063,0.16,50697,0.17
daily_screen_time_hours,95854,0.14,32788,0.11
app_opens_per_day,80710,0.12,25705,0.09
notifications_per_day,67584,0.10,34221,0.12
stress_level,55148,0.08,19626,0.07
work_study_hours,51518,0.07,27777,0.09
sleep_hours,44480,0.06,22455,0.08
academic_work_impact,44224,0.06,25721,0.09


In [3]:
#Find Categorical Labels
Categorical = ['gender', 'stress_level', 'academic_work_impact']

for column in Categorical:
    TrainCategories = set(train[column].dropna().unique())
    print(column)
    print("Categories", sorted(TrainCategories))


gender
Categories ['Female', 'Male', 'Other']
stress_level
Categories ['High', 'Low', 'Medium']
academic_work_impact
Categories ['No', 'Yes']


In [4]:
#Find Numerical Features
Numerical = ['age', 'daily_screen_time_hours', 'social_media_hours', 'gaming_hours', 'work_study_hours', 'sleep_hours', 'notifications_per_day', 'weekend_screen_time']
numericalfeatures = (train[Numerical]).describe().transpose()
numericalfeatures

,count,mean,std,min,25%,50%,75%,max
age,662440.0,26.615408,5.153162,18.00,22.00,27.00,31.00,35.00
daily_screen_time_hours,595515.0,7.640865,2.721446,0.50,5.48,7.77,9.84,15.00
social_media_hours,557374.0,2.471038,1.316137,0.00,1.45,2.31,3.37,8.00
gaming_hours,564548.0,1.459265,0.934552,0.00,0.70,1.33,2.09,4.00
work_study_hours,639851.0,2.366971,1.258797,0.00,1.36,2.20,3.20,6.00
sleep_hours,646889.0,6.804334,1.234512,4.50,5.78,6.80,7.87,9.00
notifications_per_day,623785.0,145.894900,65.917556,20.00,93.00,150.00,204.00,250.00
weekend_screen_time,579306.0,9.479866,2.856006,0.51,7.28,9.58,11.75,17.56


In [5]:
numericalcorrelation = (train[Numerical]).corrwith(train['addicted_label'])
numericalcorrelation = numericalcorrelation.sort_values(ascending=False)
numericalcorrelation

daily_screen_time_hours    0.611398
weekend_screen_time        0.589903
social_media_hours         0.532409
work_study_hours           0.251416
gaming_hours               0.205283
sleep_hours                0.042545
age                        0.004043
notifications_per_day     -0.011583
dtype: float64

In [6]:
#Categorical Features
Categorical = ['gender', 'stress_level', 'academic_work_impact']
categoricalsummaries = []

for feature in Categorical:
    categoricaldata = train[[feature, 'addicted_label']].copy()
    categoricaldata[feature] = categoricaldata[feature].fillna('missing')

    categoricalsummary = (categoricaldata.groupby(feature, dropna=False)
                          .agg(size=('addicted_label', 'size'), average=('addicted_label', 'mean'))
                          .reset_index())
    categoricalsummary = categoricalsummary.rename(columns={feature : 'category'})

    categoricalsummary.insert(0, 'feature', feature)
    categoricalsummaries.append(categoricalsummary)


categoricalsummaries = pd.concat(categoricalsummaries)    
categoricalsummaries

,feature,category,size,average
0,gender,Female,221595,0.703838
1,gender,Male,223662,0.723198
2,gender,Other,217078,0.701020
3,gender,missing,29034,0.708790
0,stress_level,High,220873,0.711431
1,stress_level,Low,207783,0.711160
2,stress_level,Medium,207565,0.705663
3,stress_level,missing,55148,0.709001
0,academic_work_impact,No,316579,0.711029
1,academic_work_impact,Yes,330566,0.707883


In [7]:
#is missing predictive
missingresults = []

features = [column for column in train.columns if column != 'id']

for feature in features:
    missingmask = train[feature].isna()

    present_addicited_label_rate = train.loc[~missingmask, 'addicted_label'].mean()
    missing_addicited_label_rate = train.loc[missingmask, 'addicted_label'].mean()

    missingresults.append({
        'feature' : feature,
        'present_addicited_label_rate' : present_addicited_label_rate,
        'missing_addicited_label_rate' : missing_addicited_label_rate
    })  
    missingdf = pd.DataFrame(missingresults)

missingdf
#missing data has no impact on addicited label

,feature,present_addicited_label_rate,missing_addicited_label_rate
0,age,0.709251,0.713402
1,daily_screen_time_hours,0.709122,0.711301
2,social_media_hours,0.709427,0.709415
3,gaming_hours,0.709182,0.710505
4,work_study_hours,0.709326,0.710645
5,sleep_hours,0.709153,0.713377
6,notifications_per_day,0.709336,0.710242
7,app_opens_per_day,0.708980,0.712787
8,weekend_screen_time,0.709187,0.710654
9,gender,0.709452,0.708790


In [8]:
features = [column for column in test.columns if column != 'id']
numericalfeatures = (
    test[features].select_dtypes(include='number').columns.tolist()
)
categoricalfeatures = (
    test[features].select_dtypes(exclude='number').columns.tolist()
)


In [18]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split

In [10]:
features = [column for column in test.columns if column != 'id']
numericalfeatures = (
    test[features].select_dtypes(include='number').columns.tolist()
)
categoricalfeatures = (
    test[features].select_dtypes(exclude='number').columns.tolist()
)

numericalpipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
    ])
categoricalpipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehotencoder', OneHotEncoder(handle_unknown='ignore'))
    ])

preprocessor = ColumnTransformer(transformers=[
    ("numerical", numericalpipeline, numericalfeatures),
    ('categorical', categoricalpipeline, categoricalfeatures)
],remainder='drop')

In [39]:
#rest of model
model = Pipeline(steps=[
    ('preprocessor', preprocessor), ('classification', LogisticRegression(class_weight='balanced'))
])

In [19]:
Xtrain, Xtest, ytrain, ytest = train_test_split(train[features], train['addicted_label'], test_size=0.2)

model.fit(Xtrain, ytrain)

ypred = model.predict(Xtest)
yprob = model.predict_proba(Xtest)[:, 1]

print("Accuracy:", accuracy_score(ytest, ypred))
print("\nConfusion Matrix:\n", confusion_matrix(ytest, ypred))
print("\nClassification Report:\n", classification_report(ytest, ypred))

Accuracy: 0.8325064726557415

Confusion Matrix:
 [[33824  6315]
 [16845 81290]]

Classification Report:
               precision    recall  f1-score   support

           0       0.67      0.84      0.74     40139
           1       0.93      0.83      0.88     98135

    accuracy                           0.83    138274
   macro avg       0.80      0.84      0.81    138274
weighted avg       0.85      0.83      0.84    138274

